# Computer Vision Notebook 4: CNN Image Classification with CIFAR-10

**Goal:** train a CNN to classify small color images into 10 categories.

CIFAR-10 is harder than MNIST because the images are color photos with backgrounds, textures, and larger variation. It is a good bridge from simple digits to real-world vision.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print('TensorFlow version:', tf.__version__)

In [ ]:
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_train = y_train.squeeze()
y_test = y_test.squeeze()

x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

print('Training images:', x_train.shape)
print('Test images:', x_test.shape)

In [ ]:
plt.figure(figsize=(10, 5))
for i in range(20):
    plt.subplot(4, 5, i+1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i]], fontsize=9)
    plt.axis('off')
plt.tight_layout()

## 1. Build a small CNN

This model is intentionally compact so it can train in a camp setting. Bigger models usually perform better but take longer.

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),

    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 2. Train on a smaller subset for fast demonstration

Use the full dataset by changing `USE_SMALL_SUBSET = False`.

In [ ]:
USE_SMALL_SUBSET = True
if USE_SMALL_SUBSET:
    train_images = x_train[:10000]
    train_labels = y_train[:10000]
else:
    train_images = x_train
    train_labels = y_train

history = model.fit(train_images, train_labels, epochs=3, batch_size=64,
                    validation_split=0.1, verbose=1)

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f'Test accuracy: {test_acc:.3f}')

plt.figure(figsize=(6, 4))
plt.plot(history.history['accuracy'], label='train accuracy')
plt.plot(history.history['val_accuracy'], label='validation accuracy')
plt.xlabel('epoch')
plt.ylabel('accuracy')
plt.legend();

## 3. Predictions

Some mistakes are understandable: a tiny 32 x 32 image can be ambiguous even for humans.

In [ ]:
probs = model.predict(x_test[:25], verbose=0)
preds = np.argmax(probs, axis=1)

plt.figure(figsize=(10, 10))
for i in range(25):
    plt.subplot(5, 5, i+1)
    plt.imshow(x_test[i])
    title = f'{class_names[preds[i]]}
true: {class_names[y_test[i]]}'
    plt.title(title, color=('green' if preds[i] == y_test[i] else 'red'), fontsize=8)
    plt.axis('off')
plt.tight_layout()

## 4. Challenge

Try adding data augmentation, increasing epochs, or making the model deeper. What changes improve validation accuracy? What changes cause overfitting?